# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library. The dataset contains ordered logistic regression outputs capturing socio-demographics, knowledge adoption, and rangeland management practices among pastoral households in Northern Kenya.

### Dataset Source

The dataset is described using a Croissant schema at:

[https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant --quiet

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and display the dataset's metadata
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}\n")
if hasattr(metadata, "keywords"):
    print(f"Keywords: {', '.join(metadata.keywords)}\n")
if hasattr(metadata, "dataCollectionTimeframe"):
    print(f"Data collection period: {metadata.dataCollectionTimeframe}\n")
if hasattr(metadata, "spatialCoverage"):
    print(f"Spatial coverage: {metadata.spatialCoverage}\n")
if hasattr(metadata, "personalSensitiveInformation"):
    print(f"Personal sensitive information: {', '.join(metadata.personalSensitiveInformation)}")

## 2. Data Overview

Let's review available record sets in the dataset and inspect their IDs and associated field IDs. All entities are referenced by their `@id` fields.

In [ ]:
# List available record sets, display their @id and field @id's

record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets found in the metadata. Please check the dataset structure.")
else:
    for rs in record_sets:
        print(f"Record set name: {rs.name}")
        print(f"  @id: {rs.id}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id})")
        print("\n")

## 3. Data Extraction

We'll load data from the available record sets into DataFrames and examine their columns. All record sets and fields are referenced using their `@id`.

In [ ]:
# Prepare to extract data from all available record sets, referenced by @id

record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for record set '@id': {rs_id}")
        if len(records) > 0:
            print(f"Columns (@id): {list(dataframes[rs_id].columns)}\n")
        else:
            print("No records found.\n")
    except Exception as e:
        print(f"Error loading records for record set '@id'={rs_id}: {e}\n")

# Pick the first record set for quick inspection
if record_set_ids:
    first_rs_id = record_set_ids[0]
    if first_rs_id in dataframes:
        display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Now we conduct basic EDA. We'll:
- Select a numeric field by its `@id`
- Filter records where this value is above a threshold
- Normalize the chosen numeric column
- If available, group by a categorical field by its `@id`

**NOTE:** If the data is empty, update the record set and field IDs below to match your dataset's structure as revealed above.

In [ ]:
# Example: Select a numeric field for analysis (adjust the @id to your dataset if empty)

# Replace these example IDs with actual ones from your data listed previously

# Fallback: Use the first record set and auto-detect a likely numeric field
import numpy as np

record_set_id = record_set_ids[0] if record_set_ids else None

if record_set_id and record_set_id in dataframes and not dataframes[record_set_id].empty:
    df = dataframes[record_set_id]
    # Attempt to choose a numeric field by dtype
    potential_numeric_fields = [col for col in df.columns if np.issubdtype(df[col].dropna().dtype, np.number)]
    if potential_numeric_fields:
        numeric_field = potential_numeric_fields[0]
        print(f"Using numeric field (@id): {numeric_field}\n")
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (mean):")
        display(filtered_df.head())

        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field}:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Attempt to group by a likely categorical field (non-numeric)
        potential_group_fields = [col for col in df.columns if not np.issubdtype(df[col].dropna().dtype, np.number)]
        if potential_group_fields:
            group_field = potential_group_fields[0]
            print(f"Grouping by field (@id): {group_field}")
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            display(grouped_df.head())
        else:
            print("No categorical group field detected.")
    else:
        print("No numeric fields detected in the selected record set.")
else:
    print("No non-empty dataframes loaded. Please check your dataset and record set IDs.")

## 5. Visualization

Let's visualize data distributions or relationships as available in the dataset. Adjust the field `@id` values as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization example: histogram for the numeric field
if record_set_id and record_set_id in dataframes and not dataframes[record_set_id].empty:
    df = dataframes[record_set_id]
    numeric_fields = [col for col in df.columns if np.issubdtype(df[col].dropna().dtype, np.number)]
    if numeric_fields:
        plt.figure(figsize=(7,4))
        sns.histplot(df[numeric_fields[0]].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of {numeric_fields[0]} (@id)")
        plt.xlabel(numeric_fields[0])
        plt.ylabel("Count")
        plt.show()
    else:
        print("No numeric field available for visualization.")
else:
    print("No data available for visualization. Please check previous steps.")

## 6. Conclusion

We loaded the dataset defined by a Croissant schema and inspected record sets, fields, and extracted data using their `@id` references. Basic exploratory analysis and a visualization step illustrate how you can approach a new Croissant-compliant dataset with `mlcroissant`. For richer analysis or domain-specific insight, refine field selection and customize data processing based on the data structure discovered.

Remember to always use `@id` references for robust, schema-aligned analysis.